# Example 7: Advection-diffusion in a cylindrical geometry

Here, we consider release of a molecule from a cylindrical wall with Pouiselle flow.

In [ ]:
import dolfin as d
import sympy as sym
import numpy as np
import pathlib
import logging
import gmsh  # must be imported before pyvista if dolfin is imported first

from smart import config, mesh, model, mesh_tools, visualization
from smart.units import unit
from smart.model_assembly import (
    Compartment,
    Parameter,
    Reaction,
    Species,
    SpeciesContainer,
    ParameterContainer,
    CompartmentContainer,
    ReactionContainer,
)

from matplotlib import pyplot as plt
import matplotlib.image as mpimg
from matplotlib import rcParams

logger = logging.getLogger("smart")
logger.setLevel(logging.INFO)

We define the relevant units here.

In [ ]:
# Aliases - base units
uM = unit.uM
um = unit.um
molecule = unit.molecule
sec = unit.sec
dimensionless = unit.dimensionless
# Aliases - units used in model
D_unit = um**2 / sec
flux_unit = uM * um / sec
vol_unit = uM
surf_unit = molecule / um**2

## Model generation

We define the compartments and species first, with their respective containers.

In [ ]:
Cyto = Compartment("Cyto", 3, um, 1)
PM = Compartment("PM", 2, um, 10,
                 vel=["0","0","100.0*[1-(x[0]**2 + x[1]**2//4)]"])

cc = CompartmentContainer()
cc.add([Cyto, PM])

A = Species("A", "1+0.1*z", vol_unit, 1.0, D_unit, "Cyto")
sc = SpeciesContainer()
sc.add([A])

Define parameters and reactions, then place in respective containers.
* r1: release of A from PM

In [ ]:
# flux to keep constant 
Awall = Parameter.from_expression("Awall", "1 + 0.1*z", vol_unit)
Jwall = Parameter("Jwall", 1000, flux_unit / vol_unit)
r1 = Reaction(
    "r1",
    [],
    ["A"],
    param_map={"Awall": "Awall", "Jwall": "Jwall"},
    eqn_f_str="Jwall*(Awall-A)",
    explicit_restriction_to_domain="PM",
)

pc = ParameterContainer()
pc.add([Awall, Jwall])
rc = ReactionContainer()
rc.add([r1])

## Create and load in mesh

Here, we consider cells embedded in a cube mesh. The source cell is located at (0,0,0) and 8 other cells are spread equidistant through the mesh.

In [ ]:
domain, facet_markers, cell_markers = mesh_tools.create_cylinders(outerRad = 2.0, innerRad=0, outerLength=10.0, innerLength=0,
                                                                  hEdge=0.5)
# Write mesh and meshfunctions to file
mesh_folder = pathlib.Path("mesh")
mesh_folder.mkdir(exist_ok=True)
mesh_path = mesh_folder / "cyl_mesh.h5"
mesh_tools.write_mesh(
    domain, facet_markers, cell_markers, filename=mesh_path
)
parent_mesh = mesh.ParentMesh(
    mesh_filename=str(mesh_path),
    mesh_filetype="hdf5",
    name="parent_mesh",
)
visualization.plot_dolfin_mesh(domain, cell_markers, facet_markers)

Initialize model and solver.

In [ ]:
config_cur = config.Config()
config_cur.flags.update({"allow_unused_components": True})
model_cur = model.Model(pc, sc, cc, rc, config_cur, parent_mesh)
config_cur.solver.update(
    {
        "final_t": 100.0,
        "initial_dt": 0.01,
        "time_precision": 8,
        "reset_timestep_for_negative_solution": True,
    }
)
model_cur.initialize()

Initialize XDMF files for saving results, save model information to .pkl file, then solve the system until `model_cur.t > model_cur.final_t`

In [ ]:
# Write initial condition(s) to file
results = dict()
result_folder = pathlib.Path(f"results")
result_folder.mkdir(exist_ok=True)
for species_name, species in model_cur.sc.items:
    results[species_name] = d.XDMFFile(
        model_cur.mpi_comm_world, str(result_folder / f"{species_name}.xdmf")
    )
    results[species_name].parameters["flush_output"] = True
    results[species_name].write(model_cur.sc[species_name].u["u"], model_cur.t)
model_cur.to_pickle("model_cur.pkl")

# Set loglevel to warning in order not to pollute notebook output
logger.setLevel(logging.WARNING)
# Solve
displayed = False
while True:
    # Solve the system
    model_cur.monolithic_solve()
    model_cur.adjust_dt()
    # Save results for post processing
    for species_name, species in model_cur.sc.items:
        results[species_name].write(model_cur.sc[species_name].u["u"], model_cur.t)

    print(f"Done with t={model_cur.t}")
    # End if we've passed the final time
    if model_cur.t >= model_cur.final_t:
        break

# plt.plot(model_cur.tvec,)

In [ ]:
u = sc["A"].sol
udiff = u - d.Expression("1.0+0.1*x[2]")